# UNITREE.md - Unitree 机器人仿真指南

基于 **unitree_mujoco** 和 **unitree_sdk2** 的完整仿真环境，支持 Go2, B2, H1, G1 等机器人。

<div align="center">
  <img src="doc/images/g1_29dof_rev.png" alt="Unitree Robot Simulation" width="600"/>
  <p><i>Unitree 机器人在 MuJoCo 中的仿真</i></p>
</div>

---

## 快速开始

### 一键安装

**脚本**: `init-unitree.sh`
自动安装 Unitree MuJoCo 仿真器及其全部依赖（C++ 版本，基于 unitree_sdk2）。


In [ ]:
# 执行安装（需要 sudo 权限）
!./init-unitree.sh


**安装内容：**

- 系统依赖：`libyaml-cpp-dev`, `libspdlog-dev`, `libboost-all-dev`, `libglfw3-dev`
- Unitree SDK2：安装至 `/opt/unitree_robotics`
- MuJoCo 3.2.6：下载至 `~/.mujoco/mujoco-3.2.6`
- 编译 `unitree_mujoco` 仿真器（支持 Go2, B2, H1, G1 等机器人）
- 编译示例程序（`stand_go2`, `climb_stairs`）

### 基础使用


In [ ]:
# 1. 启动仿真器（Go2 机器狗 + 地形场景）
!./unitree_mujoco/simulate/build/unitree_mujoco -r go2 -s scene_terrain.xml


In [ ]:

# 2. 另开终端运行控制程序
!./unitree_mujoco/example/cpp/build/stand_go2



> **注意：** Python SDK (`unitree_sdk2py`) 由于 CycloneDDS 版本兼容性问题，需手动配置。推荐使用 C++ 版本。

---

## 控制策略示例

### 1. Go2 站立控制

站起3秒后趴下（默认行为）


In [ ]:
!./unitree_mujoco/simulate/build/unitree_mujoco -r go2 -s scene.xml


In [ ]:
!./unitree_mujoco/example/cpp/build/stand_go2



### 2. B2 四足机器人仿真


In [ ]:
!./unitree_mujoco/simulate/build/unitree_mujoco -r b2 -s scene.xml


In [ ]:
!./unitree_mujoco/example/cpp/build/stand_go2  # 需修改为 B2 型号



### 3. H1 人形机器人（启用虚拟挂带）


In [ ]:
# 修改 unitree_mujoco/simulate/config.yaml:
#   robot: "h1"
#   enable_elastic_band: 1

!./unitree_mujoco/simulate/build/unitree_mujoco -r h1 -s scene.xml

# 按键操作：
#   9 = 启用/禁用挂带
#   7 = 放下机器人
#   8 = 吊起机器人


### 4. G1 人形机器人（29自由度）


In [ ]:
!./unitree_mujoco/simulate/build/unitree_mujoco -r g1 -s scene.xml


In [ ]:
!./unitree_mujoco/example/cpp/build/stand_go2  # 需使用 unitree_hg 消息



### 5. 崎岖地形测试

手动拖拽机器人测试地形适应性


In [ ]:
!./unitree_mujoco/simulate/build/unitree_mujoco -r go2 -s scene_terrain.xml
# 可手动拖拽机器人到障碍物位置


### 6. Go2 爬楼梯（Trot步态自动行走）✨NEW

基于实现的四足步态控制器，自动行走并爬楼梯。


In [ ]:
# 启动仿真器（包含10级楼梯）
!./unitree_mujoco/simulate/build/unitree_mujoco -r go2 -s scene_terrain.xml


In [ ]:

# 另开终端运行Trot步态控制器
!./unitree_mujoco/example/cpp/build/climb_stairs



**场景中的楼梯位置：**

- 楼梯1：(x=1.2~3.0, y=4.0)，10级台阶，高度0.075m~1.425m
- 楼梯2：(x=1.2~3.0, y=6.0)，10级台阶，高度0.075m~1.425m

**`climb_stairs` 控制器特性：**

| 特性               | 说明                   |
| ------------------ | ---------------------- |
| **步态类型** | Trot步态（对角腿同步） |
| **站立时间** | 0-2秒自动站起          |
| **行走开始** | 2秒后开始前进          |
| **前进速度** | 0.3 m/s                |
| **步长**     | 0.15 m                 |
| **抬腿高度** | 0.08 m（适应楼梯）     |
| **步态周期** | 0.5秒/周期             |

**实现原理：**


In [ ]:
1. Trot步态生成器
   ├─ 相位管理：FR-RL同相，FL-RR反相
   ├─ 摆动相 (0.0-0.5)：抬腿 + 前移
   └─ 支撑相 (0.5-1.0)：着地 + 推进

2. 足端轨迹规划
   ├─ X方向：周期性前后摆动
   ├─ Y方向：恒定侧向偏移
   └─ Z方向：正弦曲线抬腿

3. 逆运动学求解
   ├─ 解析式IK（髋-大腿-小腿）
   ├─ 工作空间约束
   └─ PD控制器执行


**使用建议：**

- 机器人会先站立2秒，然后自动前进
- 默认朝+X方向移动，需手动调整初始朝向对准楼梯
- 步态参数已针对Go2调优，适合低矮楼梯（<15cm台阶）
- 如需调整：修改 `climb_stairs.cpp` 中的参数

---

## 配置文件

**修改配置文件** (`unitree_mujoco/simulate/config.yaml`):


In [ ]:
robot: "go2"              # 机器人型号：go2/b2/h1/g1/b2w/go2w
robot_scene: "scene.xml"  # 场景文件：scene.xml/scene_terrain.xml
domain_id: 1              # DDS domain ID (仿真建议用1，实物用0)
interface: "lo"           # 网卡名称 (仿真用 lo，实物用 enp3s0 等)
print_scene_information: 1
enable_elastic_band: 0    # 人形机器人虚拟挂带 (H1 建议启用)


---

## 控制接口对比

| 接口类型             | 特点                   | 适用场景             |
| -------------------- | ---------------------- | -------------------- |
| **C++ SDK**    | 直接DDS通信，低延迟    | 实时控制、嵌入式部署 |
| **Python SDK** | 简洁易用（需手动安装） | 快速原型、算法验证   |
| **ROS2**       | 生态丰富、工具链完善   | 导航、SLAM、多机协作 |

---

## 支持的机器人

- **Go2** / **Go2W**: 四足机器人（12自由度）
- **B2** / **B2W**: 中型四足机器人（12自由度）
- **H1** / **H1-2**: 人形机器人（27自由度）
- **G1**: 双臂人形机器人（29自由度）

---

## 场景文件

- **`scene.xml`**: 平坦地面
- **`scene_terrain.xml`**: 崎岖地形（台阶、障碍物、楼梯）

---

## 键盘控制

在仿真器窗口中可用：

| 按键            | 功能                            |
| --------------- | ------------------------------- |
| `Space`       | 暂停/继续仿真                   |
| `Backspace`   | 重置场景                        |
| `→` / `←` | 加速/减速仿真                   |
| `9`           | 启用/禁用虚拟挂带（人形机器人） |
| `7` / `8`   | 放下/吊起机器人（虚拟挂带模式） |
| 鼠标拖拽        | 手动操控机器人关节              |

---

## Sim-to-Real（仿真到实物）

无需修改代码，仅通过命令行参数切换：


In [ ]:
# 仿真模式（默认）
!./unitree_mujoco/example/cpp/build/stand_go2


In [ ]:

# 实物模式（需连接机器人网口）
!./unitree_mujoco/example/cpp/build/stand_go2 enp3s0  # enp3s0 为网卡名



代码中通过网卡参数自动切换 `domain_id` 和网络接口：

- 无参数：`domain_id=1`, `interface="lo"` (仿真)
- 带网卡参数：`domain_id=0`, `interface=<网卡名>` (实物)

---

## ROS2 接口

**unitree_ros2** 提供完整的ROS2生态集成：

- 消息定义（`unitree_go`/`unitree_hg`）
- 服务接口（导航、感知）
- 可视化工具（rviz配置）
- 录制回放（rosbag2）

### 安装 unitree_ros2


In [ ]:
# 1. 克隆仓库（外部依赖，克隆到任意目录）
!git clone https://github.com/unitreerobotics/unitree_ros2.git


In [ ]:
!cd unitree_ros2


In [ ]:

# 2. 安装依赖并编译
!source /opt/ros/jazzy/setup.bash  # 或你的ROS2版本


In [ ]:
!./install.sh



### 编译仿真示例


In [ ]:
# Source unitree_ros2 环境（仿真模式，假设在 ~/unitree_ros2）
!source $HOME/unitree_ros2/setup_local.sh


In [ ]:
!export ROS_DOMAIN_ID=1  # 匹配仿真器的domain_id


In [ ]:

# 编译示例节点（回到项目根目录）
!cd unitree_mujoco/example/ros2


In [ ]:
!colcon build


In [ ]:

# Source编译结果
!source install/setup.bash



### 运行ROS2控制示例


In [ ]:
# 终端1: 启动仿真器
!./unitree_mujoco/simulate/build/unitree_mujoco -r go2 -s scene.xml


In [ ]:

# 终端2: 运行ROS2节点
!source $HOME/unitree_ros2/setup_local.sh


In [ ]:
!export ROS_DOMAIN_ID=1


In [ ]:
!source unitree_mujoco/example/ros2/install/setup.bash


In [ ]:
!ros2 run stand_go2 stand_go2


In [ ]:

# 终端3: 查看话题（可选）
!source $HOME/unitree_ros2/setup_local.sh


In [ ]:
!export ROS_DOMAIN_ID=1


In [ ]:
!ros2 topic list


In [ ]:
!ros2 topic echo /lowcmd



### ROS2优势

- **可视化**：rviz2实时显示机器人状态
- **录制回放**：rosbag2记录测试数据
- **模块化**：导航/感知节点即插即用
- **调试工具**：rqt图形界面参数调整

### 示例场景


In [ ]:
# 使用ROS2导航栈（需额外配置nav2）
!ros2 launch unitree_nav bringup.launch.py


In [ ]:

# 录制测试数据
!ros2 bag record -a -o test_walk


In [ ]:

# 回放分析
!ros2 bag play test_walk



---

## 强化学习（RL）部署 ✨推荐

**unitree_rl_gym** 提供官方预训练的强化学习策略，实现鲁棒的行走和运动控制。

### 工作流程


In [ ]:
Train (Isaac Gym) → Play (验证) → Sim2Sim (MuJoCo) → Sim2Real (真实机器人)


我们直接使用官方预训练模型，跳过训练步骤，在 MuJoCo 中部署。

### 环境准备


In [ ]:
# 1. 安装 rsl_rl（RL算法库）
!git clone https://github.com/leggedrobotics/rsl_rl.git


In [ ]:
!cd rsl_rl && git checkout v1.0.2


In [ ]:
!pip3 install -e .


In [ ]:
!cd ..


In [ ]:

# 2. 初始化 unitree_rl_gym 子模块
!git submodule update --init --recursive



### 运行 G1 人形机器人


In [ ]:
# 设置 Python 路径（在项目根目录执行）
!export PYTHONPATH=$PWD/unitree_rl_gym:$PYTHONPATH


In [ ]:

# 运行 G1 预训练模型（60秒演示）
!cd unitree_rl_gym


In [ ]:
!python3 -u deploy/deploy_mujoco/deploy_mujoco.py g1.yaml



**观察窗口将自动打开，G1 会执行：**

- 自动站立平衡
- 前进行走（0.5 m/s）
- IMU 姿态补偿
- 动态步态调整

### 运行其他机器人


In [ ]:
# deploy_mujoco
!python3 -u deploy/deploy_mujoco/deploy_mujoco.py h1.yaml


In [ ]:
# deploy_mujoco
!python3 -u deploy/deploy_mujoco/deploy_mujoco.py h1_2.yaml



### 挑战场景：崎岖地形与楼梯 ⛰️


In [ ]:
# deploy_mujoco
!python3 -u deploy/deploy_mujoco/deploy_mujoco.py g1_terrain.yaml


In [ ]:
# deploy_mujoco
!python3 -u deploy/deploy_mujoco/deploy_mujoco.py g1_stairs.yaml



**地形场景包含：**
- **楼梯 1** (y=3.0)：10级，高度 15cm/级（陡峭）
- **楼梯 2** (y=5.0)：8级，高度 12cm/级（缓和）
- **障碍物**：箱子、圆柱体、斜坡
- **随机岩石**：模拟自然地形

**观察要点：**
- RL 策略的动态平衡能力
- 足端轨迹的自适应调整
- 扰动恢复（可用鼠标拖拽测试）

### 自定义运动指令

修改配置文件中的 `cmd_init` 参数来改变运动行为：

**配置文件位置**: `deploy/deploy_mujoco/configs/g1.yaml`


In [ ]:
# 初始运动命令 [vx, vy, vyaw]
cmd_init: [0.5, 0, 0]  # 默认：向前走 0.5 m/s

# cmd_scale 缩放系数
cmd_scale: [2.0, 2.0, 0.25]  # [X速度, Y速度, 旋转速度]


**常用动作配置：**

| 动作               | `cmd_init` 值   | 说明           |
| ------------------ | ----------------- | -------------- |
| **向前走**   | `[0.5, 0, 0]`   | 默认前进速度   |
| **向后退**   | `[-0.5, 0, 0]`  | 后退           |
| **向右侧行** | `[0, 0.3, 0]`   | 横向移动（右） |
| **向左侧行** | `[0, -0.3, 0]`  | 横向移动（左） |
| **原地左转** | `[0, 0, 1.0]`   | 逆时针旋转     |
| **原地右转** | `[0, 0, -1.0]`  | 顺时针旋转     |
| **斜向行走** | `[0.3, 0.2, 0]` | 前进 + 向右    |
| **边走边转** | `[0.3, 0, 0.8]` | 前进 + 左转    |
| **静止平衡** | `[0, 0, 0]`     | 仅姿态保持     |

**实际速度计算：**


In [ ]:
实际vx = cmd_init[0] * cmd_scale[0]  # 例: 0.5 * 2.0 = 1.0 m/s
实际vy = cmd_init[1] * cmd_scale[1]
实际vyaw = cmd_init[2] * cmd_scale[2]  # 例: 1.0 * 0.25 = 0.25 rad/s


### 使用自定义训练模型

如果你训练了自己的模型（通过 Isaac Gym），可以替换预训练模型：


In [ ]:
# 修改 g1.yaml 中的 policy_path
policy_path: "{LEGGED_GYM_ROOT_DIR}/logs/g1/exported/policies/policy_lstm_1.pt"
# 默认预训练模型：{LEGGED_GYM_ROOT_DIR}/deploy/pre_train/g1/motion.pt


### RL 策略 vs 手工控制器

| 特性               | RL 策略            | 手工控制器 (climb_stairs) |
| ------------------ | ------------------ | ------------------------- |
| **训练方式** | 强化学习自动探索   | 人工设计步态参数          |
| **鲁棒性**   | 极强，适应各种地形 | 中等，需调参              |
| **计算开销** | 低（神经网络推理） | 极低（解析IK）            |
| **可解释性** | 黑盒               | 高（相位、IK明确）        |
| **适用场景** | 复杂地形、动态扰动 | 结构化环境、调试学习      |

### 预训练模型说明

官方提供的预训练模型位于：


In [ ]:
unitree_rl_gym/deploy/pre_train/
├── g1/motion.pt      # G1 双臂人形 (12 DOF legs)
├── h1/motion.pt      # H1 人形 (10 DOF legs)
└── h1_2/motion.pt    # H1-2 改进版


这些模型已在 Isaac Gym 中训练数百万步，具备：

- 姿态稳定性（IMU 反馈闭环）
- 自适应步态（CPG + 神经网络）
- 扰动恢复能力

---

## 常见问题

### 1. 仿真器闪退

检查 MuJoCo 软链接是否正确：


In [ ]:
!ls -la unitree_mujoco/simulate/mujoco
# 应指向: ~/.mujoco/mujoco-3.2.6


### 2. 控制无响应

确保 `domain_id` 一致（仿真器和控制程序都用1）：

- 仿真器：`config.yaml` 中设置 `domain_id: 1`
- 控制程序：无参数启动时默认使用 `domain_id=1`

### 3. 编译错误

确认 `/opt/unitree_robotics` 存在且有 SDK 文件：


In [ ]:
!ls /opt/unitree_robotics/
# 应包含: include/ lib/


### 4. G1/H1 控制失败

G1 和 H1 需使用 `unitree_hg` 消息类型（而非 `unitree_go`）。
修改代码中的消息头文件：


In [ ]:
// 错误 (Go2/B2用)
#include <unitree/idl/go2/LowCmd_.hpp>

// 正确 (G1/H1用)
#include <unitree/idl/hg/LowCmd_.hpp>


### 5. Python SDK 缺失

系统 CycloneDDS (0.10.4) 与最新 cyclonedds-python 不兼容。

**解决方案：**

- **推荐**：使用 C++ 版本（已完全可用）
- **高级**：从源码编译最新版 CycloneDDS
- **替代**：使用 Docker/Conda 隔离环境

详见 [unitree_sdk2_python](https://github.com/unitreerobotics/unitree_sdk2_python)

---

## 相关资源

- **unitree_sdk2**: [GitHub](https://github.com/unitreerobotics/unitree_sdk2)
- **unitree_sdk2_python**: [GitHub](https://github.com/unitreerobotics/unitree_sdk2_python)
- **unitree_ros2**: [GitHub](https://github.com/unitreerobotics/unitree_ros2)
- **unitree_mujoco**: [本地文档](./unitree_mujoco/readme_zh.md)
- **Unitree 官方文档**: [support.unitree.com](https://support.unitree.com/home/zh/developer)

---

## 技术架构


In [ ]:
unitree_mujoco (仿真器)
    ├─ MuJoCo 物理引擎 (3.2.6)
    ├─ unitree_sdk2 (DDS通信)
    └─ MJCF 机器人模型

控制程序
    ├─ C++ SDK (推荐)
    │   ├─ stand_go2 (站立控制)
    │   └─ climb_stairs (步态控制)
    ├─ Python SDK (需手动安装)
    └─ ROS2 (生态丰富)

仿真环境
    ├─ domain_id: 1 (仿真)
    ├─ interface: "lo" (本地回环)
    └─ DDS通信层


---

## 许可证

- **unitree_mujoco**: BSD-3-Clause
- **unitree_sdk2**: BSD-3-Clause
- **MuJoCo**: Apache License 2.0
